# FinSafe 02 — T4 GPU environment and resumable baseline
For the verified Kaggle runtime: two Tesla T4s, driver 580.178.04, CUDA 13.0.
Uses a separate Python 3.12 GPU environment. Run notebook 01 first in the same
session, or restore its project/data files. Keep Internet and GPU enabled.

Every cell logs START/DONE/FAILED, UTC timestamps and elapsed seconds. Logs are
saved under `runs/notebook-02/logs`. Local checkpoints survive process crashes,
not guaranteed Kaggle session deletion. The final cell offers a downloadable
backup. No training or HF publication starts in this notebook.


In [ ]:
import sys, os, json, subprocess, time, datetime
from pathlib import Path
started = time.monotonic()
print(f"[{datetime.datetime.now(datetime.timezone.utc).isoformat()}] START bootstrap", flush=True)
project = Path("/kaggle/working/finsafe")
if not project.exists():
    raise RuntimeError("Run notebook 01 first, or restore the project and prepared data.")
status = subprocess.check_output(["git", "-C", str(project), "status", "--porcelain", "--untracked-files=no"], text=True)
if status.strip():
    raise RuntimeError("Save tracked-file edits before updating the checkout.")
subprocess.run(["git", "-C", str(project), "pull", "--ff-only"], check=True)
sys.path.insert(0, str(project / "src"))
from finsafe.progress import Progress
progress = Progress(project / "runs/notebook-02/logs", "notebook-02")
progress.log(f"DONE bootstrap in {time.monotonic()-started:.1f}s")


In [ ]:
with progress.stage("01-runtime-and-secrets"):
    from kaggle_secrets import UserSecretsClient
    runtime_env = os.environ.copy()
    runtime_env.update(HF_TOKEN=UserSecretsClient().get_secret("HF_TOKEN"),
        HF_HOME="/kaggle/working/.cache/huggingface", HF_NAMESPACE="Finsafe-team",
        UV_CACHE_DIR="/kaggle/working/.cache/uv", UV_PYTHON_INSTALL_DIR="/kaggle/working/.python",
        UV_HTTP_TIMEOUT="180", CUDA_VISIBLE_DEVICES="0", TOKENIZERS_PARALLELISM="false",
        TRL_ONLY_MODE="1", DISABLE_UNSLOTH_FOR_TRL="1", PYTHONUNBUFFERED="1")
    revision_file = project / "runs/environment/model-revisions.json"
    if not revision_file.exists():
        raise RuntimeError("Run notebook 01's HF access check first.")
    revisions = json.loads(revision_file.read_text())
    global_sha = revisions["CohereLabs/tiny-aya-global"]
    progress.log(f"Global model revision: {global_sha}")
    if not (project / "data/processed/tatqa-v1/master.jsonl").exists():
        raise RuntimeError("Run notebook 01's TAT-QA preparation first.")
    gpu_py = str(project / ".venv-gpu/bin/python")
    uv = [sys.executable, "-m", "uv"]
    if not Path(gpu_py).exists():
        progress.run(uv + ["venv", "--python", "3.12", str(project / ".venv-gpu")], cwd=project, env=runtime_env)


## Install a matched GPU stack
This may download several GB and take several minutes. It does not change Kaggle's
Python 3.13 kernel. PyTorch 2.11.0 / torchvision 0.26.0 use the official CUDA 13.0
wheel index. Constraints prevent a silent jump to a different torch family.
Rerunning this stage reuses already installed packages and download cache.


In [ ]:
with progress.stage("02-install-pytorch"):
    progress.run(uv + ["pip", "install", "--python", gpu_py,
        "torch==2.11.0", "torchvision==0.26.0", "--index-url",
        "https://download.pytorch.org/whl/cu130"], cwd=project, env=runtime_env)


In [ ]:
with progress.stage("03-install-toolkits"):
    progress.run(uv + ["pip", "install", "--python", gpu_py, "-r", "requirements-gpu.txt",
        "-c", "constraints-kaggle-t4.txt"], cwd=project, env=runtime_env)
    progress.run(uv + ["pip", "check", "--python", gpu_py], cwd=project, env=runtime_env)
    freeze = subprocess.check_output(uv + ["pip", "freeze", "--python", gpu_py], cwd=project, env=runtime_env, text=True)
    (project / "runs/environment/gpu-freeze.txt").write_text(freeze)
    progress.log("Saved full package freeze")


In [ ]:
with progress.stage("04-gpu-and-import-smoke"):
    smoke = """import torch
from transformers import AutoTokenizer, BitsAndBytesConfig
from aligntune.core.backend_factory import create_sft_trainer
from safetune.runner.recover import ReStaTrainer
print('Torch:', torch.__version__, 'CUDA:', torch.version.cuda)
assert torch.cuda.is_available(), 'CUDA unavailable in project GPU environment'
print('Visible GPU:', torch.cuda.get_device_name(0))
x=torch.ones((128,128),device='cuda',dtype=torch.float16)
print('FP16 matrix smoke:', float((x@x)[0,0]))
print('Native BF16:', torch.cuda.is_bf16_supported())
import bitsandbytes as bnb
print('bitsandbytes:', bnb.__version__)
print('GPU imports and FP16 computation passed')
"""
    progress.run([gpu_py, "-u", "-c", smoke], cwd=project, env=runtime_env)
    progress.run([str(project / ".venv-gpu/bin/finsafe"), "doctor", "--gpu"], cwd=project, env=runtime_env)


## Token filtering and real curation
This rejects complete examples that exceed 1024 chat tokens. It never truncates
financial tables silently. Counts will be smaller than the initial 2500/200/200.
A stage reuses files only after checking the manifest against current inputs.


In [ ]:
with progress.stage("05-token-filter"):
    from finsafe.resume import sha256_file
    filtered = project / "data/processed/tatqa-tokenized"
    source = project / "data/processed/tatqa-v1/master.jsonl"
    if filtered.exists():
        manifest = json.loads((filtered / "token_manifest.json").read_text())
        if manifest["input_sha256"] != sha256_file(source) or manifest["revision"] != global_sha or manifest["max_tokens"] != 1024:
            raise RuntimeError("Token-filter settings changed. Use a fresh output directory.")
        progress.log("CHECKPOINT reused: matching token-filter manifest")
    else:
        progress.run([gpu_py, "-u", "scripts/filter_token_lengths.py", "--input", str(source),
            "--out", str(filtered), "--revision", global_sha, "--max-tokens", "1024"], cwd=project, env=runtime_env)
    progress.log((filtered / "token_manifest.json").read_text())


In [ ]:
with progress.stage("06-curation"):
    curated = filtered / "curated"
    from finsafe.progress import atomic_json
    source_hash = sha256_file(filtered / "train_alpaca.jsonl")
    marker = curated / "finsafe_source.json"
    if curated.exists():
        if not marker.exists() or json.loads(marker.read_text())["source_sha256"] != source_hash:
            raise RuntimeError("Curation folder lacks a matching completed checkpoint; use a fresh folder.")
        progress.log("CHECKPOINT reused: completed curation")
    else:
        progress.run([gpu_py, "-u", "scripts/curate.py", "--input", str(filtered / "train_alpaca.jsonl"),
            "--out", str(curated)], cwd=project, env=runtime_env)
        atomic_json(marker, {"source_sha256":source_hash})
    progress.log((curated / "manifest.json").read_text())


## Eight-example baseline smoke test
This is a model-load/format test, not the reported benchmark. Uses 4-bit NF4
with FP16 compute on GPU 0; all compared deployment conditions must use the same
quantization settings. The model weights download here. Each response is saved
atomically. Rerunning with the same settings resumes from the last saved response.


In [ ]:
with progress.stage("07-baseline-eight"):
    from finsafe.data import read_jsonl, write_jsonl
    smoke_path = project / "data/processed/tatqa-smoke.jsonl"
    write_jsonl(smoke_path, read_jsonl(filtered / "validation.jsonl")[:8])
    prediction_path = project / "runs/baseline-smoke/predictions.jsonl"
    progress.run([gpu_py, "-u", "scripts/generate.py", "--model", "CohereLabs/tiny-aya-global",
        "--revision", global_sha, "--data", str(smoke_path), "--condition", "aligned-4bit-smoke",
        "--out", str(prediction_path), "--max-new-tokens", "64", "--load-in-4bit", "--resume"], cwd=project, env=runtime_env)
    progress.run([str(project / ".venv-gpu/bin/finsafe"), "summarize", "--data", str(smoke_path),
        "--predictions", str(prediction_path), "--out", "runs/baseline-smoke/report.json"], cwd=project, env=runtime_env)
    progress.run([str(project / ".venv-gpu/bin/finsafe"), "audit-replay", "--data", str(smoke_path),
        "--predictions", str(prediction_path), "--out", "runs/baseline-smoke/auditkit.json"], cwd=project, env=runtime_env)
    progress.log("Inspect the saved generations. This smoke score is not a final benchmark result.")


## Save recovery files before ending the session
This creates a downloadable ZIP of prepared data, logs and run metadata (not
credentials, model cache or virtual environments). Download it via the link below
or retain Kaggle notebook output. Local files alone are not durable session backup.
Training later also supports optional private HF checkpoint backups when a write
token is supplied. This notebook does not upload to HF.


In [ ]:
with progress.stage("08-backup-and-finish"):
    import zipfile
    from IPython.display import FileLink, display
    backup = Path("/kaggle/working/finsafe-stage02-recovery.zip")
    with zipfile.ZipFile(backup, "w", compression=zipfile.ZIP_DEFLATED) as archive:
        for folder in [project / "data/processed", project / "runs"]:
            for file in folder.rglob("*"):
                if file.is_file() and not file.is_symlink() and not file.name.startswith(".env"):
                    archive.write(file, str(file.relative_to(project)))
    progress.log(f"Recovery ZIP ready: {backup.name} ({backup.stat().st_size/2**20:.1f} MiB)")
    display(FileLink(str(backup)))
    progress.log("Stage 02 complete. Share token-filter counts and baseline smoke output before training.")
